# 03 - Descriptive statistics and distributions
**Goal:** per-feature summaries in the course conventions (n-1 variance, Tukey quartiles, G1, G2), per genre and
per decade, plus histograms and categorical bar charts.

**Inputs:** `data/processed/tracks.parquet`, `data/processed/genre_tracks.parquet`.

**Outputs:** `reports/tables/tab_03_*.csv`, figures `fig_03_*`, `results.json -> descriptive`.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats as ss
from sonicatlas import DATA, TAB, AUDIO_FEATURES, results, theme
from sonicatlas.stats import describe, describe_frame, tukey_quartiles

theme.apply()
a = pd.read_parquet(DATA / "processed" / "tracks.parquet")
gt = pd.read_parquet(DATA / "processed" / "genre_tracks.parquet")
FOCUS = ["classical", "jazz", "country", "rock", "metal", "hip-hop", "edm", "pop"]

## Worked example: the tutorial data set 4, 5, 5, 6, 6, 6, 7, 8, 9, 14

In [2]:
tut = [4, 5, 5, 6, 6, 6, 7, 8, 9, 14]
d = describe(tut)
print({k: d[k] for k in ["mean", "median", "mode", "variance", "q1", "q3", "iqr", "q1_pandas", "q3_pandas", "g1", "g2", "pearson_kurtosis"]})
print("scipy check:", ss.skew(tut, bias=False), ss.kurtosis(tut, bias=False))

{'mean': 7.0, 'median': 6.0, 'mode': 6.0, 'variance': 8.222222222222221, 'q1': 5.0, 'q3': 8.0, 'iqr': 3.0, 'q1_pandas': 5.25, 'q3_pandas': 7.75, 'g1': 1.8026238575477382, 'g2': 3.8414118751956594, 'pearson_kurtosis': 6.841411875195659}
scipy check: 1.802623857547738 3.841411875195659


## Whole-catalogue summary (1,204,025 tracks)

In [3]:
desc = describe_frame(a, AUDIO_FEATURES)
desc.to_csv(TAB / "tab_03_01_descriptive_all.csv")
desc[["n", "mean", "median", "mode", "variance", "std", "q1", "q3", "iqr", "g1", "g2", "cv", "shape"]]

,n,mean,median,mode,variance,std,q1,q3,iqr,g1,g2,cv,shape
danceability,1204025,0.493057,0.501,0.0,0.035974,0.189669,0.356,0.633,0.277,-0.101539,-0.608492,0.384681,"roughly symmetric, near-normal tails (mesokurtic)"
energy,1204025,0.509536,0.524,0.032,0.086839,0.294684,0.252,0.766,0.514,-0.085723,-1.222906,0.578337,"roughly symmetric, flat (platykurtic)"
loudness,1204025,-11.808703,-9.791,-60.0,48.750162,6.982132,-15.254,-6.717,8.537,-1.278703,1.782713,-0.59127,"left-skewed (long tail of low values), heavy-t..."
speechiness,1204025,0.084382,0.0446,0.034,0.013454,0.115991,0.0351,0.0723,0.0372,4.243116,22.347986,1.374595,"right-skewed (long tail of high values), heavy..."
acousticness,1204025,0.446751,0.389,0.0,0.14838,0.385201,0.0376,0.861,0.8234,0.162789,-1.635658,0.862228,"roughly symmetric, flat (platykurtic)"
instrumentalness,1204025,0.282861,0.00808,0.0,0.14159,0.376284,0.000008,0.719,0.718992,0.796936,-1.175392,1.330282,"moderately right-skewed, flat (platykurtic)"
liveness,1204025,0.201599,0.125,0.111,0.032565,0.180459,0.0968,0.245,0.1482,2.266779,5.245271,0.895137,"right-skewed (long tail of high values), heavy..."
valence,1204025,0.427987,0.403,0.038,0.073162,0.270485,0.191,0.644,0.453,0.25497,-1.049935,0.631993,"roughly symmetric, flat (platykurtic)"
tempo,1204025,117.89907,116.838,120.001,925.306505,30.418851,94.294,137.047,42.753,0.398327,-0.292435,0.258008,"roughly symmetric, near-normal tails (mesokurtic)"
duration_min,1204025,4.147331,3.738983,4.0,7.308944,2.703506,2.9015,4.764,1.8625,8.042149,146.282708,0.651866,"right-skewed (long tail of high values), heavy..."


Tukey quartiles vs pandas default (linear interpolation) - on a million rows the two methods agree to 3 decimals;
the report uses Tukey values.

In [4]:
(desc[["q1", "q1_pandas", "q3", "q3_pandas"]].astype(float)).round(4)

,q1,q1_pandas,q3,q3_pandas
danceability,0.3560,0.3560,0.6330,0.6330
energy,0.2520,0.2520,0.7660,0.7660
loudness,-15.2540,-15.2540,-6.7170,-6.7170
speechiness,0.0351,0.0351,0.0723,0.0723
acousticness,0.0376,0.0376,0.8610,0.8610
instrumentalness,0.0000,0.0000,0.7190,0.7190
liveness,0.0968,0.0968,0.2450,0.2450
valence,0.1910,0.1910,0.6440,0.6440
tempo,94.2940,94.2940,137.0470,137.0470
duration_min,2.9015,2.9015,4.7640,4.7640


## Per-genre and per-decade tables

In [5]:
rows = []
for gname, grp in gt[gt.track_genre.isin(FOCUS)].groupby("track_genre"):
    for c in ["danceability", "energy", "valence", "acousticness", "loudness", "tempo"]:
        rows.append({"genre": gname, "feature": c, **{k: v for k, v in describe(grp[c]).items() if k != "shape"}})
by_genre = pd.DataFrame(rows)
by_genre.to_csv(TAB / "tab_03_02_descriptive_by_genre.csv", index=False)
by_genre.pivot(index="genre", columns="feature", values="mean").round(3)

feature,acousticness,danceability,energy,loudness,tempo,valence
genre,,,,,,
classical,0.920,0.386,0.197,-19.920,108.026,0.392
country,0.329,0.555,0.592,-7.220,124.271,0.513
edm,0.124,0.656,0.752,-5.420,121.630,0.462
hip-hop,0.198,0.730,0.695,-5.947,118.425,0.553
jazz,0.766,0.489,0.309,-12.967,115.967,0.487
metal,0.036,0.481,0.841,-4.927,129.480,0.425
pop,0.403,0.581,0.579,-7.168,121.405,0.477
rock,0.227,0.549,0.648,-6.662,121.424,0.478


In [6]:
canon = a[a.is_canonical & a.decade.notna() & (a.decade >= 1950)]
rows = []
for dec, grp in canon.groupby("decade"):
    for c in ["danceability", "energy", "valence", "acousticness", "loudness", "duration_min"]:
        rows.append({"decade": int(dec), "feature": c, **{k: v for k, v in describe(grp[c]).items() if k != "shape"}})
by_dec = pd.DataFrame(rows)
by_dec.to_csv(TAB / "tab_03_03_descriptive_by_decade.csv", index=False)
by_dec.pivot(index="decade", columns="feature", values="mean").round(3)

feature,acousticness,danceability,duration_min,energy,loudness,valence
decade,,,,,,
1950,0.834,0.427,4.124,0.296,-15.632,0.426
1960,0.698,0.440,3.789,0.354,-14.667,0.465
1970,0.531,0.482,4.220,0.452,-13.573,0.514
1980,0.472,0.479,4.256,0.468,-14.337,0.477
1990,0.553,0.465,4.189,0.411,-15.053,0.428
2000,0.461,0.494,4.108,0.508,-11.611,0.446
2010,0.399,0.497,4.184,0.547,-10.780,0.409
2020,0.309,0.560,3.687,0.587,-9.656,0.408


## Histograms with KDE

In [7]:
s = a.sample(200_000, random_state=42)
fig, axes = plt.subplots(2, 5, figsize=(15, 5.6))
units = {"loudness": "dB", "tempo": "BPM", "duration_min": "min"}
for ax, c in zip(axes.ravel(), AUDIO_FEATURES):
    x = s[c].clip(upper=s[c].quantile(0.995))
    ax.hist(x, bins=50, density=True, color=theme.OKABE_ITO[0], alpha=0.55)
    kde = ss.gaussian_kde(x.sample(20_000, random_state=42))
    xs = np.linspace(x.min(), x.max(), 200)
    ax.plot(xs, kde(xs), color="k", lw=1.4)
    ax.set_title(f"{c}  (G1={desc.loc[c, 'g1']:.2f})", fontsize=10)
    ax.set_xlabel(f"{c} ({units.get(c, '0-1')})", fontsize=8)
    ax.set_ylabel("Density", fontsize=8)
fig.suptitle("Speechiness, liveness and duration are strongly right-skewed; danceability is near-symmetric", fontweight="bold")
fig.tight_layout()
theme.save(fig, "fig_03_01_feature_histograms_kde")

WindowsPath('D:/VIT/SEM 3/EDA/project/Team_05_25BDS0318_25BDS0308_SonicAtlas/reports/figures/fig_03_01_feature_histograms_kde.png')

## Bar charts: key, mode, genres, explicit share

In [8]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
kc = a["key_name"].str.split(" ").str[0].value_counts().reindex(["C", "C#", "D", "D#", "E", "F", "F#", "G", "G#", "A", "A#", "B"])
axes[0].bar(kc.index, kc.values / 1000, color=theme.OKABE_ITO[0], edgecolor="k", lw=0.4)
top3 = kc.sort_values(ascending=False).index[:3]
axes[0].set(title=f"{top3[0]}, {top3[1]} and {top3[2]} are the most common keys", xlabel="Key (pitch class)", ylabel="Tracks (thousands)")
mc = a["mode_name"].value_counts()
axes[1].bar(mc.index, mc.values / 1000, color=[theme.OKABE_ITO[1], theme.OKABE_ITO[2]], edgecolor="k", hatch="//")
axes[1].set(title=f"{100 * mc['major'] / mc.sum():.1f}% of tracks are in a major key", xlabel="Mode", ylabel="Tracks (thousands)")
ex = canon.groupby("decade")["explicit"].mean() * 100
axes[2].plot(ex.index.astype(int), ex.values, marker="o", color=theme.OKABE_ITO[3])
axes[2].set(title=f"Explicit share: {ex.iloc[0]:.1f}% ({int(ex.index[0])}s) to {ex.iloc[-1]:.1f}% ({int(ex.index[-1])}s)", xlabel="Decade", ylabel="Explicit tracks (%)")
fig.tight_layout()
theme.save(fig, "fig_03_02_key_mode_explicit_bars")

WindowsPath('D:/VIT/SEM 3/EDA/project/Team_05_25BDS0318_25BDS0308_SonicAtlas/reports/figures/fig_03_02_key_mode_explicit_bars.png')

In [9]:
gc = gt["track_genre"].value_counts()
fig, ax = plt.subplots(figsize=(7, 3.4))
ax.hist(gc.values, bins=30, color=theme.OKABE_ITO[2], edgecolor="k", lw=0.4)
ax.set(title="After removing multi-genre duplicates genre sizes range widely",
       xlabel="Unique tracks per genre", ylabel="Number of genres")
theme.save(fig, "fig_03_03_genre_size_distribution")

WindowsPath('D:/VIT/SEM 3/EDA/project/Team_05_25BDS0318_25BDS0308_SonicAtlas/reports/figures/fig_03_03_genre_size_distribution.png')

In [10]:
results.save("descriptive", {
    "tutorial": {k: d[k] for k in ["mean", "median", "mode", "variance", "std", "q1", "q3", "iqr", "q1_pandas", "q3_pandas", "g1", "g2", "pearson_kurtosis"]},
    "all": desc.drop(columns="shape").astype(float).to_dict(orient="index"),
    "shape": desc["shape"].to_dict(),
    "major_pct": 100 * float(mc["major"] / mc.sum()),
    "top_keys": kc.sort_values(ascending=False).head(3).index.tolist(),
    "explicit_pct_by_decade": {int(k): float(v) for k, v in ex.items()},
    "explicit_pct_all": 100 * float(a.explicit.mean()),
    "genre_means": by_genre.pivot(index="genre", columns="feature", values="mean").to_dict(orient="index"),
    "decade_means": {int(k): v for k, v in by_dec.pivot(index="decade", columns="feature", values="mean").to_dict(orient="index").items()},
    "genre_size_min": int(gc.min()), "genre_size_max": int(gc.max()),
    "genre_size_min_name": gc.idxmin(), "genre_size_max_name": gc.idxmax(),
    "n_artists": int(a.artist_main.nunique()), "n_albums": int(a.album_id.nunique()),
    "canonical_since_1950": len(canon),
})